# Clario fine-tune on AMD Instinct MI300X

Gemma 4 E4B, bf16 LoRA, on the MI300X in this notebook. Every number comes from a result file and carries the GPU name.

| Step | What | Time |
|---|---|---|
| 1 | Check the GPU and ROCm torch | 1 min |
| 2 | Get the code (private repo, token asked once, never stored) | 1 min |
| 3 | Install training libraries without touching the ROCm torch | 3 min |
| 4 | Training data: upload `v3.zip` (from docs/ENRICHMENT_RU.md) | 2 min |
| 5 | **Smoke test** on 80 offline synthetic rows, 0.05 epoch | 5 to 10 min |
| 6 | **Sweep**: 4 rank:lr configs, best checkpoint each on val, winner on val, base and winner on test once | about 2 h, runs in the background |
| 7 | Results table, hardware stamp | 1 min |
| 8 | Optional: Gemma 4 26B-A4B (fits in bf16 on 192 GB) | about 2.5 h |
| 9 | Download results and the winning adapter | 1 min |

Long steps run with `nohup`, so a closed browser tab does not stop them. Re-running a step continues where it stopped.

## 1. GPU and torch

In [ ]:
!rocm-smi --showproductname --showmeminfo vram | grep -E "Card Series|Total Memory|Used Memory" 
import torch, shutil
print("torch", torch.__version__, "| HIP", torch.version.hip)
print("device", torch.cuda.get_device_name(0), "| bf16", torch.cuda.is_bf16_supported())
free, total = torch.cuda.mem_get_info(); print(f"VRAM free {free/2**30:.0f} / {total/2**30:.0f} GiB")
print(f"disk free {shutil.disk_usage('.').free/2**30:.0f} GiB")
assert torch.version.hip, "this torch is not a ROCm build"

## 2. Code
The repo is private: paste a GitHub token with read access when asked. It is used for this one command and not saved anywhere.

In [ ]:
import os, getpass, subprocess
REPO = os.path.expanduser("~/clario-amd")
URL = "github.com/akirykouski/clario-amd.git"
tok = getpass.getpass("GitHub token (read access): ")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", f"https://x-access-token:{tok}@{URL}", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://{URL}"], check=True)  # no token on disk
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q", f"https://x-access-token:{tok}@{URL}", "main"], check=True)
del tok
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
assert os.path.exists(f"{REPO}/train/sweep.sh"), "this checkout predates sweep.sh: push the latest main first"

## 3. Libraries
`torch` is pinned to the ROCm build already installed, so pip cannot swap it for a CUDA wheel.

In [ ]:
!pip freeze | grep -iE "^(torch|pytorch-triton-rocm|triton)==" > /tmp/rocm-pins.txt; cat /tmp/rocm-pins.txt
!pip install -q -U -c /tmp/rocm-pins.txt "transformers>=5.0" peft trl accelerate datasets huggingface_hub hf_transfer openai "pydantic>=2.8" "fastapi>=0.115" "httpx>=0.27"
import importlib, torch
for m in ("transformers", "peft", "trl", "accelerate", "datasets"):
    print(m, importlib.import_module(m).__version__)
assert torch.version.hip, "torch lost its ROCm build: restart the kernel and check /tmp/rocm-pins.txt"

In [ ]:
# Optional: a Hugging Face token gives faster downloads (Gemma 4 E4B loaded without one in this notebook).
import os, getpass
t = getpass.getpass("HF token (Enter to skip): ")
if t: os.environ["HF_TOKEN"] = t
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
del t

## 4. Training data
Upload `v3.zip` (the `train/data/v3/` folder from the data step) with the Jupyter upload button into your home folder, then run the cell.
It must contain `train.jsonl val.jsonl test.jsonl train_sft.jsonl val_sft.jsonl test_sft.jsonl SPLIT.md`.

In [ ]:
import os, zipfile, glob
os.makedirs(f"{REPO}/train/data", exist_ok=True)
zips = glob.glob(os.path.expanduser("~/v3*.zip")) + glob.glob("v3*.zip")
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        names = z.namelist()
        root = "" if any(n.startswith("v3/") for n in names) else "v3/"
        z.extractall(f"{REPO}/train/data/{root}")
!ls -la {REPO}/train/data/v3 && cat {REPO}/train/data/v3/SPLIT.md
!cd {REPO}/train && python -c "from prep import check_sft; print(check_sft('data/v3/train_sft.jsonl'), check_sft('data/v3/val_sft.jsonl'))"

## 5. Smoke test
80 synthetic rows made by code (no model), 0.05 epoch, then the adapter is scored on 8 val rows. Checks: LoRA target count printed, bf16 on ROCm, batched generation, result file written.

In [ ]:
%cd {REPO}/train
!python synth.py --offline --n 80 --out data-smoke/synth.jsonl && python prep.py --inputs data-smoke/synth.jsonl --data-dir data-smoke/v3
!python train_rocm.py --model google/gemma-4-E4B-it --data data-smoke/v3/train_sft.jsonl --val-data data-smoke/v3/val_sft.jsonl --out out/smoke --epochs 0.05 --bsz 8 --grad-accum 2 --results-dir "" 2>&1 | grep -vE "it/s\]$" | tail -25
!python eval.py --hf-model google/gemma-4-E4B-it --adapter out/smoke/adapter --data data-smoke/v3/val.jsonl --limit 8 --label smoke --out-dir out/smoke | tail -22

## 6. Sweep (Gemma 4 E4B)
Runs in the background. Configs `16:1e-4 16:2e-4 32:1e-4 32:2e-4` (rank:lr, alpha = rank), 3 epochs, a checkpoint every half epoch.
Selection on val only; test is scored once, on the winner. Re-run this cell after a crash: finished configs are skipped.

In [ ]:
%cd {REPO}/train
!nohup env BSZ=8 GA=2 bash sweep.sh > sweep_e4b.log 2>&1 &
!sleep 5; tail -n 5 sweep_e4b.log

In [ ]:
# progress: run any time
!tail -n 15 {REPO}/train/sweep_e4b.log
!rocm-smi --showuse --showmemuse | grep -E "GPU use|VRAM%"
!ls {REPO}/train/out/ 2>/dev/null; cat {REPO}/train/out/*/best.json 2>/dev/null | grep -E '"run"|"best"|"value"'

## 7. Results
Stamps every result file with the GPU name (from rocm-smi), then prints base vs fine-tuned on test.

In [ ]:
import json, glob, os
R = f"{REPO}/eval/results"
files = [f for f in glob.glob(f"{R}/*.json") if any(k in f for k in ("e4b", "a4b"))]
!python {REPO}/eval/hardware.py stamp {" ".join(files)}
def row(label):
    p = f"{R}/extraction_{label}.json"
    return json.load(open(p)) if os.path.exists(p) else None
keys = [("resolved_entity_f1", "F1 after HPO resolver (what the app stores)"), ("entity_f1", "F1, raw HPO id"),
        ("entity_precision", "precision"), ("entity_recall", "recall"), ("negation_accuracy", "negation accuracy"),
        ("family_history_accuracy", "family-history accuracy"), ("quote_grounding_rate", "quote grounding"),
        ("schema_valid_rate", "strict JSON valid"), ("latency_p50_s", "latency p50 (s)")]
for tag in ("e4b", "a4b"):
    b, f = row(f"base-{tag}"), row(f"ft-{tag}")
    if not (b and f): continue
    print(f"\n{tag}: test n={f['n']}  {f.get('hardware', {}).get('gpu_name', '?')}")
    print(f"{'metric':<45}{'base':>8}{'fine-tuned':>12}")
    for k, name in keys:
        fmt = lambda v: "n/a" if v is None else f"{v:.3f}"
        print(f"{name:<45}{fmt(b.get(k)):>8}{fmt(f.get(k)):>12}")
    s = json.load(open(f"{R}/sweep_{tag}.json"))
    print("sweep (val):", ", ".join(f"{r['run']} {r['value']:.3f}" for r in s["runs"]), "| winner", s["winner"], s["winner_checkpoint"])

## 8. Optional: Gemma 4 26B-A4B
Same active size per token (about 4B), much stronger base. bf16 LoRA on attention and shared layers (experts frozen). Start it only after step 6 finished.

In [ ]:
%cd {REPO}/train
!nohup env MODEL=google/gemma-4-26B-A4B-it TAG=a4b CONFIGS="32:1e-4" BSZ=4 GA=4 bash sweep.sh > sweep_a4b.log 2>&1 &
!sleep 5; tail -n 5 sweep_a4b.log

In [ ]:
!tail -n 15 {REPO}/train/sweep_a4b.log

## 9. Download
One zip with every result file, the training metrics and the winning adapters (a few hundred MB). The notebook disk may not survive the session.

In [ ]:
import os, glob, zipfile
from IPython.display import FileLink
out = os.path.expanduser("~/clario_results.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{REPO}/eval/results/*.json"):
        z.write(p, os.path.relpath(p, REPO))
    for tag in ("e4b", "a4b"):
        w = f"{REPO}/train/out/sweep-{tag}/WINNER"
        if os.path.exists(w):
            ad = open(w).read().strip(); ad = ad if os.path.isabs(ad) else f"{REPO}/train/{ad}"
            for p in glob.glob(f"{ad}/*"):
                z.write(p, f"adapters/{tag}/{os.path.basename(p)}")
            z.write(w, f"adapters/{tag}/WINNER")
    for p in glob.glob(f"{REPO}/train/sweep_*.log"):
        z.write(p, f"logs/{os.path.basename(p)}")
print(f"{os.path.getsize(out)/2**20:.0f} MB")
FileLink(os.path.relpath(out))